In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Correlation").getOrCreate()

df = spark.table("workspace.default.gold_anime")
display(df)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

df = spark.table("workspace.default.gold_anime").toPandas()
print(df.head())

**C1 — Core Correlation**

In [0]:
# Filter valid data
df = df[(df["year"] < 2026)]
scored = df[(df["score"] > 0) & (df["scored_by"] >= 1000)].copy()


pearson_r, pearson_p = stats.pearsonr(
    scored["score"], np.log(scored["members"] + 1)
)

spearman_r, spearman_p = stats.spearmanr(
    scored["score"], scored["members"]
)

print("Correlation: Score vs Popularity\n")

print(f"Pearson (linear):  r = {pearson_r:.3f}, p = {pearson_p:.4e}")
print(f"Spearman (rank):  r = {spearman_r:.3f}, p = {spearman_p:.4e}")

if abs(pearson_r) < 0.3:
    print("\nInsight: Weak relationship → Quality does NOT strongly drive popularity")
elif abs(pearson_r) < 0.6:
    print("\nInsight: Moderate relationship → Quality partially influences popularity")
else:
    print("\nInsight: Strong relationship → Quality strongly drives popularity")

**Insights:**<br>
There is a statistically significant moderate positive correlation (r ≈ 0.57) between score and popularity. This indicates that higher-quality anime tend to attract more members, but quality alone does not fully explain popularity.

**C2 — Genre-wise Correlation**

In [0]:
df_exp = scored.assign(
    genres=scored["genres"].str.split(",")
).explode("genres")

df_exp["genres"] = df_exp["genres"].str.strip()

# Keep only meaningful genres
genre_counts = df_exp["genres"].value_counts()
valid_genres = genre_counts[genre_counts >= 50].index

results = []

for g in valid_genres:
    sub = df_exp[df_exp["genres"] == g]
    
    if len(sub) < 20:
        continue

    r, p = stats.pearsonr(
        sub["score"], np.log(sub["members"] + 1)
    )

    results.append((g, round(r, 3), round(p, 4), len(sub)))

genre_corr = pd.DataFrame(
    results, columns=["Genre", "Pearson_r", "p_value", "Count"]
).sort_values("Pearson_r", ascending=False)

print("\nCorrelation by Genre\n")
print(genre_corr)

**Insights:**<br>
``The strength of the quality–popularity relationship varies by genre.``<br>
In genres like Sports / Sci-Fi, quality strongly drives popularity.<br>
In Romance, popularity is less dependent on score → likely driven by audience preference/tropes.